# Chen et al. (2025) JEF - Table 1 & 13 Replication
## Paper: "Maxing out short-term reversals in weekly stock returns"

**Authors:** Chen Chen, Andrew Cohen, Qiqi Liang, Licheng Sun  
**Journal:** Journal of Empirical Finance 82 (2025)  
**DOI:** https://doi.org/10.1016/j.jempfin.2025.101608

---

## Data Source
- Korean stock daily data from `korea_stock_chars_daily_2005_2024.sas7bdat`
- Period: 1998-01-01 to 2025-01-31
- Weekly observations: Monday to Friday

## Reference from Paper - Variable Definitions

### MAX Definition (from Paper, Section 3): 
> "Specifically, MAX is defined as the largest daily return in a week. As noted by Bali, Cakici, and Whitelaw (2011), the MAX variable is a simple and intuitive measure of investors' demand for lottery-like payoffs."

### Weekly Return Definition (from Paper):
- Weekly return = return from Monday to Friday
- Calculated as: (P_Friday - P_Monday) / P_Monday

### Week Definition:
- Week: Monday to Friday (5 trading days)
- MAX = max(daily return from Monday to Friday)

In [ ]:
# =============================================================================
# STEP 1: Load Daily Data and FF4 Weekly Data
# =============================================================================

import pandas as pd
import numpy as np
import warnings
import os
from pathlib import Path
warnings.filterwarnings('ignore')

# Get notebook directory and set working directory
NOTEBOOK_DIR = Path('/Users/younghwancho/dev/weather/analysis')
WEATHER_DIR = NOTEBOOK_DIR.parent
os.chdir(WEATHER_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Notebook directory: {NOTEBOOK_DIR}")

print("=" * 70)
print("STEP 1: Loading Daily Data and FF4 Weekly Data (2005-2024)")
print("=" * 70)

# Load daily stock data
DAILY_DATA_PATH = WEATHER_DIR / 'data/raw/korea_stock_chars_daily_2005_2024.sas7bdat'
print(f"Loading daily stock data from: {DAILY_DATA_PATH}")
df_daily = pd.read_sas(str(DAILY_DATA_PATH))

# Convert date to datetime
df_daily['date'] = pd.to_datetime(df_daily['date'])

# Filter to 2005-2024 (as specified in filename)
df_daily = df_daily[(df_daily['date'] >= '2005-01-01') & (df_daily['date'] <= '2024-12-31')]

print(f"Daily data shape: {df_daily.shape}")
print(f"Date range: {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"Columns: {list(df_daily.columns)}")
print()

# Load FF4 weekly data for risk-free rate (CD91)
FF4_PATH = WEATHER_DIR / 'data/raw/ff4_weekly_2005_2024.sas7bdat'
print(f"Loading FF4 weekly data from: {FF4_PATH}")
df_ff4 = pd.read_sas(str(FF4_PATH))
df_ff4['Date'] = pd.to_datetime(df_ff4['Date'])

# Filter FF4 to same period
df_ff4 = df_ff4[(df_ff4['Date'] >= '2005-01-01') & (df_ff4['Date'] <= '2024-12-31')]

print(f"FF4 data shape: {df_ff4.shape}")
print(f"FF4 Date range: {df_ff4['Date'].min()} to {df_ff4['Date'].max()}")
print(f"FF4 Columns: {list(df_ff4.columns)}")
print()
print("Sample daily data:")
print(df_daily.head())
print()
print("Sample FF4 data:")
print(df_ff4.head())

## How to Calculate MAX and Weekly RET from Daily Data

### MAX Calculation (from Paper):
> "MAX is defined as the largest daily return in a week."

**Formula:**
```
MAX_weekly = max(daily_return_Mon, daily_return_Tue, daily_return_Wed, daily_return_Thu, daily_return_Fri)
```

### Weekly RET Calculation:

**Formula:**
```
RET_weekly = (Price_Friday - Price_Monday) / Price_Monday
```

Or equivalently:
```
RET_weekly = (1 + r_Mon) * (1 + r_Tue) * (1 + r_Wed) * (1 + r_Thu) * (1 + r_Fri) - 1
```

In [ ]:
# =============================================================================
# STEP 2: Calculate Weekly MAX, RET, ME and EXCESS_RET from Daily Data
# =============================================================================

print("=" * 70)
print("STEP 2: Calculating Weekly MAX, RET, ME and EXCESS_RET")
print("=" * 70)

# Convert date to datetime
df_daily['date'] = pd.to_datetime(df_daily['date'])

# =============================================================================
# DATA PREPROCESSING - Empirical Finance Convention
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2A: Data Preprocessing (Empirical Finance Convention)")
print("=" * 70)

# CRITICAL: Convert returns from percentage to decimal format
print("\nConverting returns from percentage to decimal format...")
df_daily['ret'] = df_daily['ret'] / 100

# Filter valid returns (-1 < ret < 1)
print("Filtering valid returns (-1 < ret < 1)...")
df_daily = df_daily[(df_daily['ret'] > -1) & (df_daily['ret'] < 1)].copy()

# Get year and week number for grouping
df_daily['year'] = df_daily['date'].dt.isocalendar().year
df_daily['week'] = df_daily['date'].dt.isocalendar().week
df_daily['dayofweek'] = df_daily['date'].dt.dayofweek

# Filter to Monday-Friday only (0-4)
df_daily = df_daily[df_daily['dayofweek'] <= 4].copy()

print(f"Final filtered data: {len(df_daily):,} observations")

# =============================================================================
# Prepare FF4 weekly data for risk-free rate
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2B: Merging Risk-Free Rate (CD91)")
print("=" * 70)

# Convert CD91 from percentage to decimal
df_ff4['CD91_decimal'] = df_ff4['CD91'] / 100
df_ff4['year'] = df_ff4['Date'].dt.isocalendar().year
df_ff4['week'] = df_ff4['Date'].dt.isocalendar().week
df_ff4_merge = df_ff4[['year', 'week', 'CD91_decimal']].copy()

# =============================================================================
# Calculate weekly MAX
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2C: Calculating Weekly MAX")
print("=" * 70)

weekly_max = df_daily.groupby(['permno', 'year', 'week'])['ret'].max().reset_index()
weekly_max.columns = ['permno', 'year', 'week', 'MAX']

# =============================================================================
# Calculate weekly RET
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2D: Calculating Weekly RET")
print("=" * 70)

df_daily['ret_plus_1'] = df_daily['ret'].fillna(0) + 1
weekly_ret = df_daily.groupby(['permno', 'year', 'week'])['ret_plus_1'].prod().reset_index()
weekly_ret.columns = ['permno', 'year', 'week', 'RET']
weekly_ret['RET'] = weekly_ret['RET'] - 1

# =============================================================================
# Calculate weekly ME (for value-weighted portfolios)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2E: Calculating Weekly ME")
print("=" * 70)

# Only keep non-null ME values
df_daily_me = df_daily[df_daily['ME'].notna() & (df_daily['ME'] > 0)]
weekly_me = df_daily_me.groupby(['permno', 'year', 'week'])['ME'].mean().reset_index()
weekly_me.columns = ['permno', 'year', 'week', 'ME']

# =============================================================================
# Merge all weekly data
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2F: Merging Weekly Data")
print("=" * 70)

df_weekly = pd.merge(weekly_max, weekly_ret, on=['permno', 'year', 'week'], how='inner')
df_weekly = pd.merge(df_weekly, weekly_me, on=['permno', 'year', 'week'], how='inner')
df_weekly = pd.merge(df_weekly, df_ff4_merge, on=['year', 'week'], how='left')

# Calculate excess return (RET - RF)
df_weekly['EXCESS_RET'] = df_weekly['RET'] - df_weekly['CD91_decimal']

# Week-end date
friday_dates = df_daily[df_daily['dayofweek'] == 4][['permno', 'year', 'week', 'date']].copy()
friday_dates = friday_dates.rename(columns={'date': 'week_end_date'})
df_weekly = pd.merge(df_weekly, friday_dates, on=['permno', 'year', 'week'], how='left')

print(f"Final weekly data: {len(df_weekly):,} observations")

## Reference from Paper - Table 1 Structure

### Table 1 Panel A (from Paper, Section 4.1):
> "In Panel A of our Table 1, we form quintile portfolios by sorting stocks on the MAX variable from the past week at t − 1. We report both equal-weighted and value-weighted average weekly portfolio returns from week t."

- Sort stocks into 5 quintiles based on **MAX from week t-1** (`MAX_t_minus_1`)
- Calculate average return in **week t**
- Report Q1 (low MAX) to Q5 (high MAX)

### Table 1 Panel B (from Paper, Section 4.1):
> "In Panel B, we form quintile portfolios by sorting stocks on the MAX from week t − 2. We skip week t − 1 and report the average weekly portfolio returns from week t."

- Sort stocks into 5 quintiles based on **MAX from week t-2** (`MAX_t_minus_2`)
- Calculate average return in **week t** (skip week t-1)
- Report Q1 (low MAX) to Q5 (high MAX)

In [ ]:
# =============================================================================
# STEP 3: Prepare Lagged MAX Variables and Winsorization
# =============================================================================

print("=" * 70)
print("STEP 3: Creating Lagged MAX Variables & Winsorization")
print("=" * 70)

# Sort by stock and date
df_weekly = df_weekly.sort_values(['permno', 'week_end_date']).reset_index(drop=True)

# Create lagged MAX variables
df_weekly['MAX_t_minus_1'] = df_weekly.groupby('permno')['MAX'].shift(1)
df_weekly['MAX_t_minus_2'] = df_weekly.groupby('permno')['MAX'].shift(2)

print("Lagged MAX variables created:")
print(f"  - MAX_t_minus_1: MAX from previous week")
print(f"  - MAX_t_minus_2: MAX from 2 weeks ago")

# =============================================================================
# WINSORIZATION - Empirical Finance Convention
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3B: Winsorization (1% - 99%)")
print("=" * 70)
print("Standard practice in empirical finance: winsorize at 1% and 99%")
print("This removes extreme outliers that could skew results")

def winsorize(series, lower_pct=1, upper_pct=99):
    """Winsorize a series at given percentiles."""
    lower = np.percentile(series.dropna(), lower_pct)
    upper = np.percentile(series.dropna(), upper_pct)
    return series.clip(lower=lower, upper=upper)

# Winsorize MAX and RET at 1% and 99%
print("\nBefore winsorization:")
print(f"  MAX: [{df_weekly['MAX'].min():.4f}, {df_weekly['MAX'].max():.4f}]")
print(f"  RET: [{df_weekly['RET'].min():.4f}, {df_weekly['RET'].max():.4f}]")

df_weekly['MAX'] = winsorize(df_weekly['MAX'], 1, 99)
df_weekly['MAX_t_minus_1'] = winsorize(df_weekly['MAX_t_minus_1'], 1, 99)
df_weekly['MAX_t_minus_2'] = winsorize(df_weekly['MAX_t_minus_2'], 1, 99)
df_weekly['RET'] = winsorize(df_weekly['RET'], 1, 99)

print("\nAfter winsorization (1% - 99%):")
print(f"  MAX: [{df_weekly['MAX'].min():.4f}, {df_weekly['MAX'].max():.4f}]")
print(f"  RET: [{df_weekly['RET'].min():.4f}, {df_weekly['RET'].max():.4f}]")

print()
print(df_weekly[['permno', 'week_end_date', 'MAX', 'MAX_t_minus_1', 'MAX_t_minus_2', 'RET']].head(10))

## Do We Need CD91 for Table 1?

**Answer: NO**

For **Table 1 Panel A and Panel B** (quintile sorting with Q1-Q5 only):
- We use **raw returns (RET)**, not excess returns
- No risk-free rate (CD91) needed

CD91 would only be needed for:
1. **Excess returns**: RET - CD91
2. **Factor model alphas**: regress excess returns on factors

Since we're only replicating the left panel (quintile Q1-Q5) without alphas, **CD91 is not needed**.

In [ ]:
# =============================================================================
# STEP 4: Table 1 - Quintile Portfolio Sorts on MAX (EW and VW, Excess Returns)
# =============================================================================

"""
TABLE 1 VERIFICATION - Paper vs Implementation:

Paper (Section 4.1, Chen et al. 2025):
- Panel A: Sort on MAX from week t-1 → avg weekly excess return from week t
- Panel B: Sort on MAX from week t-2 (skip week t-1) → avg weekly return from week t
- Both equal-weighted AND value-weighted average weekly portfolio returns
- Uses: Excess returns (Stock Return - Risk-free rate)
- t-stat: Newey-West (1987, 1994) with automatic lag selection

Our Implementation:
✓ MAX = max(daily return in week)
✓ Weekly RET = cumulative product (1+r1)*...*(1+r5) - 1
✓ EXCESS_RET = RET - CD91 (risk-free rate)
✓ Quintile sorting: 5 groups based on MAX
✓ Equal-weighted: simple average
✓ Value-weighted: weighted by market equity (ME)
✓ Newey-West HAC standard errors with auto lag selection
✓ Period: 2005-2024
"""

def compute_nw_stats(returns):
    """
    Compute mean and Newey-West t-statistic.
    
    Paper uses: Newey-West (1987, 1994) adjusted t-statistics with automatic lag selection
    Formula: maxlag = floor(4*(T/100)^(2/9)) where T = number of weeks
    """
    import statsmodels.api as sm
    
    T = len(returns)
    if T <= 1:
        return None
    
    # Automatic lag selection (as per paper)
    maxlag = int(np.floor(4 * (T / 100) ** (2/9)))
    maxlag = min(maxlag, T - 1)
    
    X = np.ones(T)
    
    try:
        # Newey-West HAC standard errors
        model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlag})
        return {
            'mean': model.params[0],
            'nw_t': model.params[0] / model.bse[0],
            'nw_se': model.bse[0],
            'n_weeks': T,
            'maxlag': maxlag
        }
    except:
        # Fallback
        std = np.std(returns, ddof=1)
        return {
            'mean': np.mean(returns),
            'nw_t': np.mean(returns) / (std / np.sqrt(T)),
            'nw_se': std / np.sqrt(T),
            'n_weeks': T,
            'maxlag': 0
        }


def calculate_quintile_returns_nw(df, max_col, ret_col='EXCESS_RET', vw_col=None):
    """
    Calculate quintile portfolio returns with Newey-West t-statistics.
    Supports both equal-weighted and value-weighted portfolios.
    
    Parameters:
    -----------
    df : DataFrame
        Weekly stock data
    max_col : str
        Column name for MAX variable
    ret_col : str
        Column name for return variable (default: 'EXCESS_RET')
    vw_col : str
        Column name for value-weight (e.g., 'ME')
    
    Returns:
    --------
    dict : EW and VW returns with NW t-statistics for each quintile
    """
    
    results_ew = {i: [] for i in range(1, 6)}  # Q1-Q5 Equal-weighted
    results_vw = {i: [] for i in range(1, 6)}  # Q1-Q5 Value-weighted
    
    # Group by week
    for date, group in df.groupby('week_end_date'):
        # Drop missing values
        group_clean = group.dropna(subset=[max_col, ret_col])
        
        if vw_col:
            group_clean = group_clean.dropna(subset=[vw_col])
        
        if len(group_clean) < 50:  # Need sufficient stocks for quintiles
            continue
        
        try:
            # Assign quintiles (1=low MAX, 5=high MAX)
            group_clean = group_clean.copy()
            group_clean['quintile'] = pd.qcut(group_clean[max_col], q=5, labels=[1,2,3,4,5], duplicates='drop')
            
            for q in range(1, 6):
                q_data = group_clean[group_clean['quintile'] == q]
                if len(q_data) > 0:
                    # Equal-weighted: simple average
                    results_ew[q].append(q_data[ret_col].mean())
                    
                    # Value-weighted: weighted by market equity
                    if vw_col and vw_col in q_data.columns:
                        weights = q_data[vw_col].values
                        returns = q_data[ret_col].values
                        if np.sum(weights) > 0:
                            vw_return = np.sum(returns * weights) / np.sum(weights)
                            results_vw[q].append(vw_return)
            
        except Exception as e:
            continue
    
    # Compute NW statistics
    stats = {'ew': {}, 'vw': {}}
    
    for q in range(1, 6):
        # Equal-weighted
        ew_returns = np.array(results_ew[q])
        if len(ew_returns) > 1:
            stats['ew'][q] = compute_nw_stats(ew_returns)
        
        # Value-weighted
        vw_returns = np.array(results_vw.get(q, []))
        if len(vw_returns) > 1:
            stats['vw'][q] = compute_nw_stats(vw_returns)
    
    return stats


# Panel A: Sort on MAX from week t-1
# Reference from Paper: "In Panel A of our Table 1, we form quintile portfolios 
# by sorting stocks on the MAX variable from the past week at t − 1."

print("=" * 70)
print("TABLE 1 PANEL A: Sort on MAX from week t-1")
print("=" * 70)
print("Reference: 'In Panel A of our Table 1, we form quintile portfolios by sorting")
print("           stocks on the MAX variable from the past week at t − 1.'")
print("           (Section 4.1)")
print()

# Drop rows with missing lagged MAX, ME, and EXCESS_RET
df_pa = df_weekly.dropna(subset=['MAX_t_minus_1', 'ME', 'EXCESS_RET'])
print(f"Observations for Panel A: {len(df_pa):,}")

# Calculate both EW and VW
stats_pa_ew = calculate_quintile_returns_nw(df_pa, 'MAX_t_minus_1', 'EXCESS_RET', None)
stats_pa_vw = calculate_quintile_returns_nw(df_pa, 'MAX_t_minus_1', 'EXCESS_RET', 'ME')

print("\nEqual-Weighted (Excess Returns):")
print("-" * 60)
print(f"{'Quintile':<12} {'Mean Return (%)':<18} {'NW t-stat':<12}")
print("-" * 60)

panel_a_ew_results = {}
for q in range(1, 6):
    if q in stats_pa_ew['ew'] and stats_pa_ew['ew'][q]:
        s = stats_pa_ew['ew'][q]
        mean_pct = s['mean'] * 100
        panel_a_ew_results[q] = {'mean': mean_pct, 't': s['nw_t'], 'n': s['n_weeks']}
        print(f"Q{q} (Low→High MAX): {mean_pct:>10.3f}%    {s['nw_t']:>10.2f}")

print("\nValue-Weighted (Excess Returns):")
print("-" * 60)
print(f"{'Quintile':<12} {'Mean Return (%)':<18} {'NW t-stat':<12}")
print("-" * 60)

panel_a_vw_results = {}
for q in range(1, 6):
    if q in stats_pa_vw['vw'] and stats_pa_vw['vw'][q]:
        s = stats_pa_vw['vw'][q]
        mean_pct = s['mean'] * 100
        panel_a_vw_results[q] = {'mean': mean_pct, 't': s['nw_t'], 'n': s['n_weeks']}
        print(f"Q{q} (Low→High MAX): {mean_pct:>10.3f}%    {s['nw_t']:>10.2f}")

In [ ]:
# Panel B: Sort on MAX from week t-2 (skip one week)
print()
print("=" * 70)
print("TABLE 1 PANEL B: Sort on MAX from week t-2 (skip week t-1)")
print("=" * 70)
print("Reference: 'we form quintile portfolios by sorting stocks on the MAX from week t - 2. We skip week t - 1 and report the average weekly portfolio returns from week t.'")
print("t-stat: Newey-West (1987, 1994) with automatic lag selection")
print()

# Drop rows with missing lagged MAX
df_pb = df_weekly.dropna(subset=['MAX_t_minus_2', 'ME', 'EXCESS_RET'])
print(f"Observations for Panel B: {len(df_pb):,}")

# Use EXCESS_RET
stats_pb_ew = calculate_quintile_returns_nw(df_pb, 'MAX_t_minus_2', 'EXCESS_RET', None)  # Equal-weighted
stats_pb_vw = calculate_quintile_returns_nw(df_pb, 'MAX_t_minus_2', 'EXCESS_RET', 'ME')  # Value-weighted

print("\nEqual-Weighted (Excess Returns):")
print("-" * 70)
print(f"{'Quintile':<12} {'Mean Return':<15} {'NW t-stat':<12} {'N weeks':<10}")
print("-" * 70)

panel_b_ew = {}
for q in range(1, 6):
    if q in stats_pb_ew['ew'] and stats_pb_ew['ew'][q]:
        s = stats_pb_ew['ew'][q]
        mean_pct = s['mean'] * 100
        panel_b_ew[q] = {'mean': mean_pct, 't': s['nw_t'], 'n_weeks': s['n_weeks']}
        print(f"Q{q}: {mean_pct:>8.3f}%    {s['nw_t']:>10.2f}    {s['n_weeks']:>8}")

print("\nValue-Weighted (Excess Returns):")
print("-" * 70)
print(f"{'Quintile':<12} {'Mean Return':<15} {'NW t-stat':<12} {'N weeks':<10}")
print("-" * 70)

panel_b_vw = {}
for q in range(1, 6):
    if q in stats_pb_vw['vw'] and stats_pb_vw['vw'][q]:
        s = stats_pb_vw['vw'][q]
        mean_pct = s['mean'] * 100
        panel_b_vw[q] = {'mean': mean_pct, 't': s['nw_t'], 'n_weeks': s['n_weeks']}
        print(f"Q{q}: {mean_pct:>8.3f}%    {s['nw_t']:>10.2f}    {s['n_weeks']:>8}")

## Table 1 Interpretation

### From Paper (Section 4.1):
> "Taken together, the results from Table 1 indicate that there is strong evidence that the MAX anomaly is highly significant in weekly stock returns and the effect seems to persist for at least two weeks after portfolio formation."

### Key Finding from Paper:
- High-MAX stocks underperform low-MAX stocks
- This is the "MAX anomaly" - lottery-like stocks (high MAX) have lower returns
- Effect persists for at least 2 weeks (Panel B shows this)

In [ ]:
# =============================================================================
# STEP 5: Table 1 Visualization - Following Paper Format
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.table as mtable
from pathlib import Path

NOTEBOOK_DIR = Path('/Users/younghwancho/dev/weather/analysis')

# Create figure with 2x2 subplots (Panel A/B × EW/VW)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Helper function to create table
def create_table(ax, data, title, col_labels):
    ax.axis('off')
    table_data = []
    for row in data:
        table_data.append([str(x) for x in row])
    
    table = ax.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)
    
    # Style header
    for i in range(len(col_labels)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(color='white', weight='bold')
    
    ax.set_title(title, fontsize=11, fontweight='bold', pad=15)

# Panel A: Sort on MAX from week t-1
# Equal-Weighted
ew_data_pa = []
for q in range(1, 6):
    if q in panel_a_ew:
        r = panel_a_ew[q]
        ew_data_pa.append([f'Q{q}', f'{r["mean"]:.3f}', f'({r["t"]:.2f})'])

# Value-Weighted  
vw_data_pa = []
for q in range(1, 6):
    if q in panel_a_vw:
        r = panel_a_vw[q]
        vw_data_pa.append([f'Q{q}', f'{r["mean"]:.3f}', f'({r["t"]:.2f})'])

create_table(axes[0, 0], ew_data_pa, 'Panel A: Sort on MAX from week t-1\nEqual-Weighted', ['MAX', 'Return (%)', 't-stat'])
create_table(axes[0, 1], vw_data_pa, 'Panel A: Sort on MAX from week t-1\nValue-Weighted', ['MAX', 'Return (%)', 't-stat'])

# Panel B: Sort on MAX from week t-2
# Equal-Weighted
ew_data_pb = []
for q in range(1, 6):
    if q in panel_b_ew:
        r = panel_b_ew[q]
        ew_data_pb.append([f'Q{q}', f'{r["mean"]:.3f}', f'({r["t"]:.2f})'])

# Value-Weighted
vw_data_pb = []
for q in range(1, 6):
    if q in panel_b_vw:
        r = panel_b_vw[q]
        vw_data_pb.append([f'Q{q}', f'{r["mean"]:.3f}', f'({r["t"]:.2f})'])

create_table(axes[1, 0], ew_data_pb, 'Panel B: Sort on MAX from week t-2\nEqual-Weighted', ['MAX', 'Return (%)', 't-stat'])
create_table(axes[1, 1], vw_data_pb, 'Panel B: Sort on MAX from week t-2\nValue-Weighted', ['MAX', 'Return (%)', 't-stat'])

plt.suptitle('Table 1: Returns of Weekly Portfolios Sorted on MAX\n(Chen et al. 2025 JEF Replication - Korean Stocks, 2005-2024)\nExcess Returns (RET - CD91), Newey-West t-statistics', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'table1_visualization.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\nTable 1 visualization saved to: {NOTEBOOK_DIR / 'table1_visualization.png'}")

## Summary

### What We Calculated:
1. **MAX** = max(daily return within Monday-Friday week)
2. **Weekly RET** = cumulative return: (1+r1)*(1+r2)*...*(1+r5) - 1

### Do We Need CD91?
- **Table 1 (left panel - quintile sorting): NO**
- We use raw returns, not excess returns
- CD91 would only be needed for excess returns or factor alphas

### Next Steps:
- Add market equity (ME) data for value-weighted portfolios
- Add control variables for Table 13 regression